In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.ticker as ticker
import warnings
import os

# Ignore warnings
warnings.filterwarnings("ignore")

# ================= 1. Configuration Area =================

# --- Style Configuration ---
CONFIG = {
    'label_size': 55,        # [Size] Y-axis title font size
    'x_label_size': 62,      # [Size] X-axis title (MIC_S / K_S) font size
    'tick_size': 54,         # [Size] Axis tick number size
    'annotation_size': 32,   # [Size] Mean formula text font size
    'star_size': 54,         # [Size] Significance star (***) font size
    
    'border_width': 5,       # [Thickness] Axis border and tick mark linewidth
    'plot_linewidth': 5,     # [Thickness] Violin and boxplot outline linewidth
    'ref_line_width': 6,     # [Thickness] Significance test horizontal linewidth
    'ref_line_color': 'black', # [Color] Significance horizontal line color
    
    'major_tick_length': 14,  # [Length] Axis major tick length
    
    'mean_label_dx': 0.12,   # [Position] Horizontal offset of the mean text box relative to the center
    
    'y_tick_step': 1,        # [Spacing] Y-axis major tick step size
    
    'single_fig_size': (18, 18), # [Size] Width and height of a single canvas (inches)
    
    'y_label_x_pos': -0.10   # [Position] Horizontal offset of the Y-axis title
}

# --- Color Configuration ---
colors_mic = ['#6BAED6', '#3182BD', '#08519C'] 
colors_k   = ['#FB6A4A', '#CB181D', '#67000D'] 
box_edge_color = 'dimgray'    
violin_edge_color = '#D3D3D3' 

# --- File Paths ---
# Use relative paths for GitHub compatibility
file_path = "./simulated_msc_physiological.xlsx"
save_dir = "./results_physiological"
save_name_mic = "MSC_Analysis_MICs.png" 
save_name_k = "MSC_Analysis_Ks.png"     

# --- Global Settings ---
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = CONFIG['tick_size']                
plt.rcParams['axes.linewidth'] = CONFIG['border_width']        
plt.rcParams['xtick.major.width'] = CONFIG['border_width']     
plt.rcParams['ytick.major.width'] = CONFIG['border_width']     
plt.rcParams['xtick.major.size'] = CONFIG['major_tick_length'] 
plt.rcParams['ytick.major.size'] = CONFIG['major_tick_length'] 
plt.rcParams['mathtext.fontset'] = 'stix'


# ================= 2. Statistical Analysis Function =================
def add_stat_annotation(ax, data_list, x_positions, y_start, step_h):
    combinations = []
    for i in range(len(data_list)):
        for j in range(i + 1, len(data_list)):
            combinations.append((i, j))
    
    p_values = []
    for i, j in combinations:
        t_stat, p = stats.ttest_ind(data_list[i], data_list[j], nan_policy='omit', equal_var=False)
        p_values.append(p)
    
    reject, p_corrected, _, _ = multipletests(p_values, method='holm')
    comb_info = list(zip(combinations, p_corrected))
    comb_info.sort(key=lambda x: abs(x[0][0] - x[0][1])) 
    
    current_y = y_start
    for (i, j), p_val in comb_info:
        x1, x2 = x_positions[i], x_positions[j]
        
        if p_val < 0.001: sig_text = "***"
        elif p_val < 0.01: sig_text = "**"
        elif p_val < 0.05: sig_text = "*"
        else: sig_text = "ns"
            
        line_y = current_y
        h = step_h * 0.15 # [Size] Height of vertical short lines at the ends of the significance line
        ax.plot([x1, x1, x2, x2], [line_y, line_y+h, line_y+h, line_y], 
                lw=CONFIG['ref_line_width'], c=CONFIG['ref_line_color']) 
        
        # [Position Adjustment] Decrease text_gap to move stars closer to the line
        text_gap = h * 0.1 if sig_text == "ns" else -0.08 # [Spacing] Vertical distance between text and horizontal line
        ax.text((x1+x2)/2, line_y + h + text_gap, sig_text, ha='center', va='bottom', 
                fontsize=CONFIG['star_size'], 
                color='black', fontweight='bold')
        
        current_y += step_h # [Spacing] Vertical start height increment for the next set of significance markers

# ================= 3. Core Plotting Function =================

def plot_raincloud(ax, data_df, col_indices, group_labels, colors, x_axis_label, ylim_top, stat_start_y, stat_step_h):
    
    data_list = []
    for col_idx in col_indices:
        col_data = data_df.iloc[:, col_idx].dropna()
        col_data = col_data[col_data != 0] 
        data_list.append(col_data)
    
    positions = np.arange(1, len(data_list) + 1)
    
    # --- Layer 1: Violin (Cloud) ---
    # [Size] widths=0.7 controls the maximum width of the violin
    parts = ax.violinplot(data_list, positions=positions, showmeans=False, showmedians=False, showextrema=False, widths=0.7)
    for pc in parts['bodies']:
        pc.set_facecolor('none')        
        pc.set_edgecolor(violin_edge_color) 
        pc.set_linewidth(CONFIG['plot_linewidth']) 
        pc.set_alpha(1)                
        pc.set_zorder(5)             

    # --- Layer 2: Scatter (Rain) ---
    np.random.seed(42)
    for i, d in enumerate(data_list):
        y = d.values
        x_jitter = np.random.normal(positions[i], 0.06, size=len(y)) # [Size] Controls horizontal jitter amplitude of data points
        # [Size] s=360 controls the size of scatter circles
        ax.scatter(x_jitter, y, alpha=0.4, s=360, color=colors[i], edgecolors='white', linewidth=0.5, zorder=10)

    # --- Layer 3: Boxplot (Box) ---
    # [Size] widths=0.15 controls box width
    box = ax.boxplot(data_list, positions=positions, widths=0.15, 
               patch_artist=True, 
               boxprops=dict(facecolor='none', color=box_edge_color, linewidth=CONFIG['plot_linewidth']),
               medianprops=dict(color='black', linewidth=CONFIG['plot_linewidth']),
               whiskerprops=dict(color=box_edge_color, linewidth=CONFIG['plot_linewidth']),
               capprops=dict(color=box_edge_color, linewidth=CONFIG['plot_linewidth']),
               showfliers=False, zorder=15)

    # --- Layer 4: Mean & Label ---
    for i, d in enumerate(data_list):
        mean_val = np.mean(d.values)
        # [Size] markersize=18 controls the size of the mean red dot
        ax.plot(positions[i], mean_val, 'o', markersize=18, color='#A50F15', markeredgecolor='white', zorder=20)
        
        # [Font Settings] Force numbers to use Arial font
        ax.text(positions[i] + CONFIG['mean_label_dx'], mean_val, 
                rf"$\hat{{\mu}}$ = {mean_val:.2f}", 
                fontfamily='Arial',
                # [Spacing] pad=0.2 controls distance between text and border
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.8, lw=1.5),
                va='center', ha='left',  
                zorder=25, fontsize=CONFIG['annotation_size'])

    add_stat_annotation(ax, data_list, positions, stat_start_y, stat_step_h) 

    # --- Axis Settings ---
    ax.set_xticks(positions)
    new_labels = []
    for label, d in zip(group_labels, data_list):
        n_count = len(d)
        new_labels.append(f"{label}\n(n={n_count})")
    ax.set_xticklabels(new_labels, fontsize=CONFIG['tick_size'])
    
    # [Spacing] labelpad=15 controls distance between axis title and axis line
    ax.set_ylabel("Minimum Selective Concentration (MSC)", fontsize=CONFIG['label_size'])
    ax.set_xlabel(x_axis_label, fontsize=CONFIG['x_label_size'], labelpad=15, fontweight='bold')
    
    ax.set_ylim(bottom=0, top=ylim_top)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(CONFIG['y_tick_step']))
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(CONFIG['border_width'])

# ================= 4. Main Execution =================
try:
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    df = pd.read_excel(file_path, engine='openpyxl')
    
    all_values = []
    for col_idx in [1, 2, 3, 4, 5, 6]:
        col_data = df.iloc[:, col_idx].dropna()
        col_data = col_data[col_data != 0]
        all_values.extend(col_data.values)
    
    global_max = np.max(all_values)
    global_range = global_max - np.min(all_values)
    
    # [Spacing Control] 0.3/0.02/0.08 are proportional coefficients to control top whitespace for annotations
    unified_ylim_top = global_max + global_range * 0.3
    unified_stat_start_y = global_max + global_range * 0.02
    unified_stat_step_h  = global_range * 0.08
    
    # ---------------- Plot 1: MIC_S ----------------
    fig1, ax1 = plt.subplots(figsize=CONFIG['single_fig_size'])
    plot_raincloud(ax1, df, [1, 2, 3], ["2.0-4.0", "4.0-6.0", "6.0-8.0"], colors_mic, r"$MIC_S$",
                   unified_ylim_top, unified_stat_start_y, unified_stat_step_h)
    ax1.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5) 
    
    path_mic = os.path.join(save_dir, save_name_mic)
    plt.savefig(path_mic, dpi=600, bbox_inches='tight')
    
    # ---------------- Plot 2: K_S ----------------
    fig2, ax2 = plt.subplots(figsize=CONFIG['single_fig_size'])
    plot_raincloud(ax2, df, [4, 5, 6], ["4.0-4.6", "4.6-5.3", "5.3-6.0"], colors_k, r"$K_S$",
                   unified_ylim_top, unified_stat_start_y, unified_stat_step_h)
    ax2.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)
    
    path_k = os.path.join(save_dir, save_name_k)
    plt.savefig(path_k, dpi=600, bbox_inches='tight')
    
    print(f"Figures successfully saved to: {save_dir}")
    plt.show()
    
except Exception as e:
    print(f"An error occurred: {e}")